# Introduction to Spark: PySpark Essentials (companion notebook)

This notebook follows the slides **`spark_intro_3h.pptx`** concept by concept. Every code example from the slides is here, so you can run it, change it and see what happens. Concept numbers (1–20) match the slide titles.

| Part | Time | Concepts |
|---|---|---|
| 1 · Spark basics | 0:00–0:55 | 1 Why Spark · 2 How Spark runs · 3 DataFrames and types · 4 Lazy evaluation · 5 Shuffles |
| 2 · Working with DataFrames | 0:55–2:25 | 6 Reading · 7 Columns · 8 NULLs · 9 Casting · 10 Time zones · *break* · 11 Joins · 12 Aggregations · 13 Windows |
| 3 · SQL, interop and pipelines | 2:25–3:00 | 14 Spark SQL · 15 Results · 16 Arrow · 17 Writing · 18 Lakehouse tables · 19 Pipeline patterns · 20 Errors |

**Data:** the MetroRide April trips file, `../bronze/trips_2026-04.csv` (132 trips).

**Requirements:** Java 17 or newer, and `pip install pyspark "pandas<3"` (or `uv add pyspark "pandas<3"`). Run the notebook from the `phases/phase-1-batch/pyspark/` folder.

> Cells that demonstrate an **error** catch it and print only the error class, so the notebook runs from top to bottom. Remove the `try`/`except` to see the full traceback. Spark may still print the error in red in its own log: that is normal.

*Try it* exercises are spread through the notebook; solutions are at the end.

---
# Part 1 · Spark basics

## Before class: setup check

Java must be installed, because Spark runs on the JVM. The `!` runs a shell command from the notebook.

In [1]:
!java -version

openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


## 1 · Why Spark?

Spark splits one job across many CPU cores or machines, so data larger than one computer's memory can still be processed, and the same code runs on a laptop or a cluster.

Our MetroRide data is tiny (269 trips in phase 1): pandas or DuckDB would be enough. We use Spark here to **learn the patterns that scale**.

## 2 · How Spark runs

* The **driver** runs your Python program, builds the plan and splits it into tasks.
* **Executors** run the tasks; each task processes one **partition** of the data.
* A **cluster manager** (YARN, Kubernetes, standalone) gives the driver its executors.
* `master("local[*]")` runs the driver and executors in one process on your laptop, with one task slot per CPU core.

PySpark is a Python API: your Python code sends instructions through the **Py4J** bridge to the Spark engine, which runs in a Java virtual machine (JVM). `SparkSession` is the entry point.

In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("spark-intro")
         .config("spark.sql.session.timeZone", "Asia/Bangkok")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")      # hide INFO and WARN messages

print(spark.version)
print(spark.sparkContext.defaultParallelism)  # task slots = CPU cores in local mode

4.2.0
2


In [3]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



**Spark UI.** While the session runs, open the address below in a browser to see jobs, stages and tasks. Run a few cells later in the notebook and refresh the page.

In [4]:
print(spark.sparkContext.uiWebUrl)

http://localhost:4040


## 3 · A DataFrame: rows, columns, schema, partitions

A DataFrame looks like a pandas table, but it is split into **partitions** that live on executors, and it is **immutable**: every operation returns a new DataFrame. The schema string declares column names and types. `NULL` is Spark's missing value (Python `None`).

In [5]:
trips = spark.createDataFrame(
    [("T1", "Siam", 148.0),
     ("T2", "Asok", None),
     ("T3", "Ari", 95.5)],
    "trip_id string, zone string, fare double")

trips.printSchema()
trips.show()
print("partitions:", trips.rdd.getNumPartitions())

root
 |-- trip_id: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- fare: double (nullable = true)



+-------+----+-----+
|trip_id|zone| fare|
+-------+----+-----+
|     T1|Siam|148.0|
|     T2|Asok| NULL|
|     T3| Ari| 95.5|
+-------+----+-----+

partitions: 2


### Spark types and Python types

Spark infers a type from each Python value. The defaults are not always what you want: `int` becomes `long` (64-bit) and `Decimal` becomes `decimal(38,18)`.

In [6]:
from datetime import datetime, date
from decimal import Decimal

spark.createDataFrame(
    [("Siam", 3, 1.5, Decimal("148.00"), datetime(2026, 4, 1, 8, 3), date(2026, 4, 1), True)],
    ["s", "i", "f", "d", "ts", "dt", "b"],
).printSchema()

root
 |-- s: string (nullable = true)
 |-- i: long (nullable = true)
 |-- f: double (nullable = true)
 |-- d: decimal(38,18) (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- dt: date (nullable = true)
 |-- b: boolean (nullable = true)



In pipelines, **declare the schema explicitly** with `StructType`:

In [7]:
from pyspark.sql import types as T

schema = T.StructType([
    T.StructField("trip_id", T.StringType()),
    T.StructField("fare_thb", T.DecimalType(10, 2)),
])
spark.createDataFrame([("T1", Decimal("148.00"))], schema).printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- fare_thb: decimal(10,2) (nullable = true)



### The example data

From here on we use the real April file. Every column is read as text; Concept 6 explains each option.

In [8]:
from pathlib import Path
from pyspark.sql import functions as F

DATA = Path("../bronze/trips_2026-04.csv")
assert DATA.exists(), "Run this notebook from the phases/phase-1-batch/pyspark/ folder"

COLS = ["trip_id", "driver_id", "service_zone", "requested_at", "accepted_at",
        "picked_up_at", "completed_at", "cancelled_at", "status", "fare_thb"]
csv_schema = T.StructType([T.StructField(c, T.StringType()) for c in COLS])

raw = (spark.read
       .option("header", True)
       .option("enforceSchema", False)
       .schema(csv_schema)
       .csv(str(DATA)))
raw.show(3)

+---------------+---------+------------+--------------------+--------------------+--------------------+--------------------+------------+---------+--------+
|        trip_id|driver_id|service_zone|        requested_at|         accepted_at|        picked_up_at|        completed_at|cancelled_at|   status|fare_thb|
+---------------+---------+------------+--------------------+--------------------+--------------------+--------------------+------------+---------+--------+
|BKK-20260401-M1|     D002|        Asok|2026-04-01T08:03:...|2026-04-01T08:05:...|2026-04-01T08:11:...|2026-04-01T08:24:...|        NULL|completed|  148.00|
|BKK-20260401-M2|     D003|         Ari|2026-04-01T08:20:...|2026-04-01T08:23:...|2026-04-01T08:31:...|2026-04-01T08:45:...|        NULL|completed|  161.00|
|BKK-20260401-E1|     D004|         Ari|2026-04-01T18:03:...|2026-04-01T18:04:...|2026-04-01T18:15:...|2026-04-01T18:30:...|        NULL|completed|  166.00|
+---------------+---------+------------+------------------

## 4 · Transformations are lazy; actions run jobs

**Transformations** (`select`, `withColumn`, `filter`, `join`, `groupBy`, …) only add steps to a plan. **Actions** (`show`, `count`, `first`, `collect`, `toPandas`, `toArrow`, `write`) make Spark run the plan as a **job**.

The first two lines below return immediately; the file is only read at `count()`.

In [9]:
completed = raw.filter(F.col("status") == "completed")   # transformation
fares = completed.select("trip_id", "fare_thb")           # transformation
# ... nothing has been read yet ...
fares.count()                                             # action: Spark runs a job

124

### Plans and caching

`explain()` prints the plan Spark will run. Look for **PushedFilters**: the Catalyst optimiser moved our filter into the file scan. **ReadSchema** shows that only 3 of the 10 columns are read.

In [10]:
fares.explain()

== Physical Plan ==
*(1) Project [trip_id#27, fare_thb#36]
+- *(1) Filter (isnotnull(status#35) AND (status#35 = completed))
   +- FileScan csv [trip_id#27,status#35,fare_thb#36] Batched: false, DataFilters: [isnotnull(status#35), (status#35 = completed)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/…, PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,completed)], ReadSchema: struct<trip_id:string,status:string,fare_thb:string>




Every action re-runs the plan from the source. `cache()` keeps a DataFrame in memory after its first use, which helps when one DataFrame feeds several actions.

In [11]:
fares.cache()
print(fares.count())      # job 1: reads the CSV and fills the cache
fares.show(3)             # reuses the cache
_ = fares.unpersist()

124
+---------------+--------+
|        trip_id|fare_thb|
+---------------+--------+
|BKK-20260401-M1|  148.00|
|BKK-20260401-M2|  161.00|
|BKK-20260401-E1|  166.00|
+---------------+--------+
only showing top 3 rows


## 5 · Narrow vs. wide transformations

* **Narrow** (`select`, `filter`, `withColumn`, `cast`): each output partition depends on one input partition. No data moves.
* **Wide** (`join`, `groupBy`, window functions, `distinct`, `orderBy`): rows with the same key must meet, so Spark **shuffles** data between partitions.

In a plan, a shuffle appears as an **Exchange** step. Compare the two plans:

In [12]:
raw.filter(F.col("status") == "cancelled").select("trip_id").explain()   # narrow: no Exchange

== Physical Plan ==
*(1) Project [trip_id#27]
+- *(1) Filter (isnotnull(status#35) AND (status#35 = cancelled))
   +- FileScan csv [trip_id#27,status#35] Batched: false, DataFilters: [isnotnull(status#35), (status#35 = cancelled)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/…, PartitionFilters: [], PushedFilters: [IsNotNull(status), EqualTo(status,cancelled)], ReadSchema: struct<trip_id:string,status:string>




In [13]:
raw.groupBy("service_zone").count().explain()                            # wide: Exchange hashpartitioning

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[service_zone#29], functions=[count(1)])
   +- Exchange hashpartitioning(service_zone#29, 200), ENSURE_REQUIREMENTS, [plan_id=184]
      +- HashAggregate(keys=[service_zone#29], functions=[partial_count(1)])
         +- FileScan csv [service_zone#29] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/…, PartitionFilters: [], PushedFilters: [], ReadSchema: struct<service_zone:string>




`spark.sql.shuffle.partitions` sets how many partitions a shuffle produces. The default 200 is far too many for small data.

In [14]:
print(spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "4")

200


---
# Part 2 · Working with DataFrames

## 6 · Reading CSV files

| Option | Meaning |
|---|---|
| `header` | first line = column names |
| `schema(...)` | use these names and types |
| `inferSchema` | guess types (reads the data an extra time) |
| `enforceSchema=False` | check that the header matches the schema |
| blank cell | read as `NULL` |

We read every column as text, so blank cells of cancelled trips become `NULL`:

In [15]:
raw.filter("status = 'cancelled'").select("trip_id", "picked_up_at", "fare_thb").show(3)

+---------------+------------+--------+
|        trip_id|picked_up_at|fare_thb|
+---------------+------------+--------+
|BKK-20260405-M1|        NULL|    NULL|
|BKK-20260406-E3|        NULL|    NULL|
|BKK-20260409-E1|        NULL|    NULL|
+---------------+------------+--------+
only showing top 3 rows


With `inferSchema=True`, Spark guesses types. The fare becomes `double`, which is not exact for money; that is one reason the bronze layer keeps the raw text.

In [16]:
spark.read.csv(str(DATA), header=True, inferSchema=True).select("trip_id", "requested_at", "fare_thb").printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- requested_at: timestamp (nullable = true)
 |-- fare_thb: double (nullable = true)



### Other formats and whole files

| Format | Read with | Schema comes from |
|---|---|---|
| CSV | `spark.read.csv(path, header=True)` | you, or `inferSchema` |
| Parquet | `spark.read.parquet(path)` | the file itself |
| JSON | `spark.read.json(path)` | inferred from the data |
| Whole files | `spark.read.format("binaryFile")` | path, length, content, … |

`binaryFile` reads each file as one row with its bytes in `content`. A SHA-256 hash of those bytes is a fingerprint: the same bytes always give the same hash.

In [17]:
sha = (spark.read.format("binaryFile").load(str(DATA))
         .select(F.sha2("content", 256).alias("sha256"))
         .first()["sha256"])
sha

'ee6286bcdafff7d1e5f0c771084ffc42bed1122db62155cad008a0167ad13821'

## 7 · Column expressions

`F.col("fare")` is an **expression**, not a value: it describes a computation that Spark runs later. Everybody imports the functions module as `F`.

| Method / function | Does |
|---|---|
| `select(...)` | choose or compute columns |
| `withColumn(name, expr)` | add or replace one column |
| `filter` / `where` | keep rows where the condition is True |
| `drop(...)` | remove columns |
| `F.col`, `F.lit` | a column, a constant |
| `F.when(...).otherwise(...)` | if / else |
| `alias(name)` | name a result |

In [18]:
t = (raw
  .select("trip_id",
          F.trim("service_zone").alias("zone"),
          F.lower("status").alias("status"), "fare_thb")
  .withColumn("fare", F.col("fare_thb").cast("double"))
  .withColumn("price_band",
      F.when(F.col("fare") >= 170, "high")
       .when(F.col("fare") >= 140, "mid")
       .otherwise("low"))
  .filter(F.col("zone") != "Silom")
  .drop("fare_thb"))
t.show(4)

+---------------+----+---------+-----+----------+
|        trip_id|zone|   status| fare|price_band|
+---------------+----+---------+-----+----------+
|BKK-20260401-M1|Asok|completed|148.0|       mid|
|BKK-20260401-M2| Ari|completed|161.0|       mid|
|BKK-20260401-E1| Ari|completed|166.0|       mid|
|BKK-20260401-E3|Siam|completed|160.0|       mid|
+---------------+----+---------+-----+----------+
only showing top 4 rows


### Combining conditions: `&`, `|`, `~`

Python's `and`, `or` and `not` do not work on columns. Use `&`, `|` and `~`, and put **each comparison in its own parentheses**, because `&` binds tighter than `==`.

In [19]:
# Wrong: missing parentheses
try:
    raw.filter(F.col("status") == "completed" & F.col("service_zone") == "Ari")
except Exception as e:
    print(type(e).__name__, "->", str(e)[:200])

PySparkValueError -> [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.


In [20]:
# Right: one pair of parentheses per comparison
print(raw.filter((F.col("status") == "completed")
                 & (F.col("service_zone") == "Ari")).count())
print(raw.filter(~F.col("service_zone").isin("Siam", "Asok")).count())

31
67


## 8 · NULL means "unknown"

Comparing `NULL` with anything gives `NULL`, not `False`. `filter` keeps only rows where the condition is **True**, so a row with `NULL` disappears from both a filter **and** its negation.

| Expression | Result |
|---|---|
| `NULL > 100` | `NULL` |
| `TRUE AND NULL` | `NULL` |
| `FALSE AND NULL` | `FALSE` |
| `TRUE OR NULL` | `TRUE` |
| `NOT NULL` | `NULL` |

In [21]:
df = spark.createDataFrame([("T1", 148.0), ("T2", None)], "trip_id string, fare double")
df.select("trip_id", "fare", (F.col("fare") > 100).alias("fare_gt_100")).show()

print(df.filter(F.col("fare") > 100).count(), "   # fare > 100")
print(df.filter(~(F.col("fare") > 100)).count(), "   # NOT (fare > 100): T2 is in neither!")
print(df.filter(F.col("fare").isNull()).count(), "   # fare IS NULL")

+-------+-----+-----------+
|trip_id| fare|fare_gt_100|
+-------+-----+-----------+
|     T1|148.0|       true|
|     T2| NULL|       NULL|
+-------+-----+-----------+



1    # fare > 100
0    # NOT (fare > 100): T2 is in neither!


1    # fare IS NULL


## 9 · Parsing and casting: `cast` vs. `try_cast`

Spark 4 runs in **ANSI SQL mode** by default: `cast` or `to_timestamp` on a malformed value stops the whole job. The `try_` versions return `NULL` instead, so bad values can be filtered out later.

Datetime patterns use these letters:

| Pattern | Matches |
|---|---|
| `yyyy-MM-dd` | 2026-04-01 |
| `'T'` | the literal letter T |
| `HH:mm:ss` | 08:03:00 (24-hour) |
| `XXX` | +07:00 offset |

In [22]:
print("ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))

TS = "yyyy-MM-dd'T'HH:mm:ssXXX"
bad = spark.createDataFrame(
    [("2026-04-01T08:03:00+07:00", "148.00"), ("not a time", "abc")],
    "ts string, fare string")

bad.select(
    F.try_to_timestamp("ts", F.lit(TS)).alias("ts"),
    F.col("fare").try_cast("decimal(10,2)").alias("fare")).show()

ANSI mode: true


+-------------------+------+
|                 ts|  fare|
+-------------------+------+
|2026-04-01 08:03:00|148.00|
|               NULL|  NULL|
+-------------------+------+



In [23]:
# A plain cast fails on 'abc'
try:
    bad.select(F.col("fare").cast("decimal(10,2)")).show()
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0][:160])

NumberFormatException -> [CAST_INVALID_INPUT] The value 'abc' of the type "STRING" cannot be cast to "DECIMAL(10,2)" because it is malformed. Correct the value as per the syntax, or cha


## 10 · Time zones

A timestamp is an **instant**. Spark displays it, and computes `to_date()` and `hour()`, in the **session time zone**. The same instant is 31 March at 23:30 in UTC but 1 April at 06:30 in Bangkok.

In [24]:
one = (spark.createDataFrame(
           [("2026-04-01T06:30:00+07:00",)], "raw string")
         .select(F.to_timestamp("raw", TS).alias("ts")))

for tz in ["UTC", "Asia/Bangkok"]:
    spark.conf.set("spark.sql.session.timeZone", tz)
    print(tz)
    one.select("ts", F.to_date("ts").alias("date"),
               F.hour("ts").alias("hour")).show()

spark.conf.set("spark.sql.session.timeZone", "Asia/Bangkok")   # back to Bangkok

UTC


+-------------------+----------+----+
|                 ts|      date|hour|
+-------------------+----------+----+
|2026-03-31 23:30:00|2026-03-31|  23|
+-------------------+----------+----+

Asia/Bangkok


+-------------------+----------+----+
|                 ts|      date|hour|
+-------------------+----------+----+
|2026-04-01 06:30:00|2026-04-01|   6|
+-------------------+----------+----+



---
## Break · Try it (exercise 1)

1. Build a `price_band` column (like in Concept 7) for trips in **your favourite zone** only.
2. Count how many April trips have a `NULL` fare. Which status do they have?

In [25]:
# Exercise 1: your code here

## 11 · Joins

`df.join(other, on, how)`:

| `how` | Keeps |
|---|---|
| `inner` | rows with a match on both sides |
| `left` | all left rows; `NULL`s where there is no match |
| `right` / `full` | all right rows / all rows |
| `left_semi` | left rows that **have** a match (no right columns) |
| `left_anti` | left rows that have **no** match |

`left_anti` is central to the PySpark tutorial: it finds rows that are **not yet** in a table. Remember that `NULL` keys never match.

In [26]:
trips_d = spark.createDataFrame([("T1", "D1"), ("T2", "D2"), ("T3", "D9")], "trip_id string, driver_id string")
drivers = spark.createDataFrame([("D1", "Siam"), ("D2", "Ari")], "driver_id string, home_zone string")

for how in ["inner", "left", "left_semi", "left_anti"]:
    print(f'how="{how}"')
    trips_d.join(drivers, "driver_id", how).show()

how="inner"


+---------+-------+---------+
|driver_id|trip_id|home_zone|
+---------+-------+---------+
|       D1|     T1|     Siam|
|       D2|     T2|      Ari|
+---------+-------+---------+

how="left"


+---------+-------+---------+
|driver_id|trip_id|home_zone|
+---------+-------+---------+
|       D1|     T1|     Siam|
|       D9|     T3|     NULL|
|       D2|     T2|      Ari|
+---------+-------+---------+

how="left_semi"


+---------+-------+
|driver_id|trip_id|
+---------+-------+
|       D1|     T1|
|       D2|     T2|
+---------+-------+

how="left_anti"


+---------+-------+
|driver_id|trip_id|
+---------+-------+
|       D9|     T3|
+---------+-------+



## 12 · Aggregations

`groupBy(...)` followed by `agg(...)` with one or more aggregate functions. Cast fares to `decimal` before summing so money stays exact. Aggregates skip `NULL`s, except `count("*")`, which counts rows.

In [27]:
(raw.groupBy("service_zone")
    .agg(F.count("*").alias("trips"),
         F.count_if(F.col("status") == "completed").alias("completed"),
         F.sum(F.col("fare_thb").cast("decimal(10,2)")).alias("fare_thb"),
         F.countDistinct("driver_id").alias("drivers"))
    .orderBy("service_zone")
    .show())

+------------+-----+---------+--------+-------+
|service_zone|trips|completed|fare_thb|drivers|
+------------+-----+---------+--------+-------+
|         Ari|   33|       31| 5676.00|     12|
|        Asok|   32|       30| 5325.00|     12|
|        Siam|   33|       32| 5560.00|     12|
|       Silom|   34|       31| 6014.00|     12|
+------------+-----+---------+--------+-------+



### Try it (exercise 2)

For **completed** trips, compute the average **passenger pickup wait** in minutes (`picked_up_at - requested_at`) per `service_zone`. Hint: parse both columns with `F.try_to_timestamp(..., F.lit(TS))`, then use `F.unix_timestamp(...)`, which gives seconds.

In [28]:
# Exercise 2: your code here

## 13 · Window functions

A window computes a value for each row from a **group of related rows**, without collapsing them like `groupBy` does.

* `partitionBy`: which rows belong together.
* `orderBy`: the order inside each group.
* `row_number()`: 1, 2, 3, … per group. Keeping row 1 removes duplicates.

In [29]:
from pyspark.sql import Window

ver = spark.createDataFrame(
    [("T1", "completed", 1), ("T1", "cancelled", 2), ("T2", "completed", 1)],
    "trip_id string, status string, version int")

w = Window.partitionBy("trip_id").orderBy(F.desc("version"))
ver.withColumn("rn", F.row_number().over(w)).show()

+-------+---------+-------+---+
|trip_id|   status|version| rn|
+-------+---------+-------+---+
|     T1|cancelled|      2|  1|
|     T1|completed|      1|  2|
|     T2|completed|      1|  1|
+-------+---------+-------+---+



In [30]:
latest = (ver.withColumn("rn", F.row_number().over(w))
             .filter("rn = 1").drop("rn"))      # one row per trip: the newest version
latest.show()

+-------+---------+-------+
|trip_id|   status|version|
+-------+---------+-------+
|     T1|cancelled|      2|
|     T2|completed|      1|
+-------+---------+-------+



### Row numbers in file order

`monotonically_increasing_id()` gives unique, increasing ids, but **not consecutive**: the partition number is stored in the high bits, so the second partition starts at 8589934592. A `row_number()` window ordered by it gives 1, 2, 3, …

A window **without** `partitionBy` moves every row into one partition: fine for one small file, a bottleneck on big data.

In [31]:
r = (spark.range(6).repartition(2)
       .withColumn("mono_id", F.monotonically_increasing_id()))
file_order = Window.orderBy("mono_id")
r.withColumn("row_no", F.row_number().over(file_order)).show()

+---+----------+------+
| id|   mono_id|row_no|
+---+----------+------+
|  2|         0|     1|
|  5|         1|     2|
|  0|8589934592|     3|
|  1|8589934593|     4|
|  3|8589934594|     5|
|  4|8589934595|     6|
+---+----------+------+



### Try it (exercise 3)

For each `driver_id`, keep only their **latest** April request (largest `requested_at`). How many rows do you get?

In [32]:
# Exercise 3: your code here

---
# Part 3 · SQL, interop and pipelines

## 14 · Spark SQL and temporary views

Register a DataFrame as a **temporary view** and query it with SQL. SQL and the DataFrame API go through the same optimiser, so both give the same plan and result. A temp view lives only in this `SparkSession`; it is a name for a DataFrame, not a stored table.

In [33]:
raw.createOrReplaceTempView("trips")

spark.sql('''
  SELECT service_zone, count(*) AS trips FROM trips
  WHERE status = 'cancelled' GROUP BY service_zone
  ORDER BY service_zone''').show()

+------------+-----+
|service_zone|trips|
+------------+-----+
|         Ari|    2|
|        Asok|    2|
|        Siam|    1|
|       Silom|    3|
+------------+-----+



In [34]:
(raw.filter(F.col("status") == "cancelled")
    .groupBy("service_zone").agg(F.count("*").alias("trips"))
    .orderBy("service_zone").show())

+------------+-----+
|service_zone|trips|
+------------+-----+
|         Ari|    2|
|        Asok|    2|
|        Siam|    1|
|       Silom|    3|
+------------+-----+



### Try it (exercise 4)

Rewrite exercise 2 (average pickup wait per zone) as **one Spark SQL query** on the `trips` view. Spark SQL has `try_to_timestamp(col, fmt)` and `unix_timestamp(ts)` too.

In [35]:
# Exercise 4: your code here

## 15 · Getting results out

| Action | Returns | Size |
|---|---|---|
| `show(n, truncate=False)` | printed table | first n rows |
| `show(vertical=True)` | one block per row | first n rows |
| `first()` | one `Row` | 1 row |
| `collect()` | list of `Row` | **all** rows |
| `toPandas()` | pandas DataFrame | **all** rows |
| `toArrow()` | pyarrow Table | **all** rows |

`collect`, `toPandas` and `toArrow` pull every row into the **driver's memory**. That is fine for 132 rows and dangerous for 132 million: use `limit()` or aggregate first.

In [36]:
print(raw.first()["trip_id"])
print(raw.select("trip_id", "status").limit(2).collect())

BKK-20260401-M1
[Row(trip_id='BKK-20260401-M1', status='completed'), Row(trip_id='BKK-20260401-M2', status='completed')]


In [37]:
raw.select("trip_id", "status", "fare_thb").show(2, truncate=False)
raw.limit(1).show(vertical=True)

+---------------+---------+--------+
|trip_id        |status   |fare_thb|
+---------------+---------+--------+
|BKK-20260401-M1|completed|148.00  |
|BKK-20260401-M2|completed|161.00  |
+---------------+---------+--------+
only showing top 2 rows


-RECORD 0----------------------------
 trip_id      | BKK-20260401-M1      
 driver_id    | D002                 
 service_zone | Asok                 
 requested_at | 2026-04-01T08:03:... 
 accepted_at  | 2026-04-01T08:05:... 
 picked_up_at | 2026-04-01T08:11:... 
 completed_at | 2026-04-01T08:24:... 
 cancelled_at | NULL                 
 status       | completed            
 fare_thb     | 148.00               



## 16 · Arrow: Spark ↔ Python tables

Apache Arrow is a columnar in-memory format shared by Spark, pandas, Polars, DuckDB and pyducklake. With Arrow enabled, `toPandas()` and `createDataFrame()` are much faster, and Spark 4 adds `toArrow()`.

Watch the types: timestamps leave Spark in **UTC**, and pandas receives `Decimal` objects for decimal columns.

In [38]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

small = raw.select(
    "trip_id",
    F.try_to_timestamp("requested_at", F.lit(TS)).alias("requested_at"),
    F.col("fare_thb").try_cast("decimal(10,2)").alias("fare_thb"))

tbl = small.toArrow()                  # pyarrow.Table
print(type(tbl).__name__, tbl.num_rows, "rows")
print(tbl.schema)

Table 132 rows
trip_id: string
requested_at: timestamp[us, tz=UTC]
fare_thb: decimal128(10, 2)


In [39]:
pdf = small.toPandas()                 # pandas DataFrame
print(pdf.dtypes)

back = spark.createDataFrame(tbl)      # Arrow -> Spark
back.printSchema()

trip_id                 object
requested_at    datetime64[ns]
fare_thb                object
dtype: object
root
 |-- trip_id: string (nullable = true)
 |-- requested_at: timestamp (nullable = true)
 |-- fare_thb: decimal(10,2) (nullable = true)



## 17 · Writing data with Spark

`df.write` with a **mode** and a **format**. Spark writes a **folder**, not a file: one part file per partition, plus a `_SUCCESS` marker. `partitionBy` adds `key=value` sub-folders.

| mode | If the path exists |
|---|---|
| `error` (default) | fail |
| `append` | add new files |
| `overwrite` | replace everything |
| `ignore` | do nothing |

In [40]:
import shutil

OUT = Path("spark_intro_out")
shutil.rmtree(OUT, ignore_errors=True)          # start clean on every run

(raw.select("trip_id", "service_zone", "status")
    .write.mode("overwrite").partitionBy("service_zone")
    .parquet(str(OUT / "trips_parquet")))

for p in sorted((OUT / "trips_parquet").rglob("*")):
    if p.is_file() and not p.name.startswith("."):      # skip Spark's .crc checksum files
        print(p.relative_to(OUT))

trips_parquet/_SUCCESS
trips_parquet/service_zone=Ari/part-00000-e7e64e0c-1575-4d51-9be6-bdaa2710fa9b.c000.snappy.parquet
trips_parquet/service_zone=Asok/part-00000-e7e64e0c-1575-4d51-9be6-bdaa2710fa9b.c000.snappy.parquet
trips_parquet/service_zone=Siam/part-00000-e7e64e0c-1575-4d51-9be6-bdaa2710fa9b.c000.snappy.parquet
trips_parquet/service_zone=Silom/part-00000-e7e64e0c-1575-4d51-9be6-bdaa2710fa9b.c000.snappy.parquet


In [41]:
# The default mode is "error": writing to an existing path fails
try:
    raw.select("trip_id").write.parquet(str(OUT / "trips_parquet"))
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0][:70], "...")

AnalysisException -> [PATH_ALREADY_EXISTS] Path file:/… ...


## 18 · From files to lakehouse tables

A folder of Parquet files is **not a table**: there are no transactions and no snapshots, and nothing stops the same data from being written twice. Watch what `append` does when a job is accidentally run twice:

In [42]:
folder = str(OUT / "appended")
for run in [1, 2]:                                   # the same job runs twice
    raw.select("trip_id", "status").write.mode("append").parquet(folder)
    total = spark.read.parquet(folder).count()
    print(f"after run {run}: {total} rows")

after run 1: 132 rows


after run 2: 264 rows


A **table format** (Apache Iceberg, Delta Lake, DuckLake) adds a metadata layer: each write becomes one atomic **snapshot**, old snapshots stay readable, and writers can be coordinated. With Iceberg or Delta, Spark plugs the format in as a catalog (`spark.sql.catalog.<name>`) and writes with `df.writeTo("lake.silver.trips").append()`.

In the PySpark tutorial, Spark does all the processing and **pyducklake** commits each Spark DataFrame to DuckLake (through Arrow) as a snapshot. The duplicate guards in the next concept are what stop the double append above.

## 19 · Pipeline patterns in PySpark

Four patterns that the tutorial builds from today's concepts. Here they run on small in-memory examples.

**File guard:** skip a file whose hash was already loaded.

In [43]:
loaded_hashes = spark.createDataFrame([(sha,)], "file_sha256 string")   # pretend April was loaded
already = loaded_hashes.filter(F.col("file_sha256") == sha)
print("skip the file" if already.count() > 0 else "load the file")

skip the file


**Row guard:** keep the first copy of each key inside a batch, then only keys that are not loaded yet.

In [44]:
existing = raw.select("trip_id")                                  # rows already loaded
batch = spark.createDataFrame(
    [("BKK-20260401-M1", 1),     # already loaded
     ("NEW-1", 2), ("NEW-1", 3), # new, but sent twice
     ("NEW-2", 4)],
    "trip_id string, row_no int")

first_copy = Window.partitionBy("trip_id").orderBy("row_no")
new_rows = (batch
    .withColumn("copy_no", F.row_number().over(first_copy))
    .filter(F.col("copy_no") == 1).drop("copy_no")
    .join(existing, "trip_id", "left_anti"))
new_rows.orderBy("row_no").show()

+-------+------+
|trip_id|row_no|
+-------+------+
|  NEW-1|     2|
|  NEW-2|     4|
+-------+------+



**Incremental + idempotent:** process only rows that are not in the target yet, so running twice changes nothing. **Validation:** bad values become `NULL` with `try_cast`, then fail the filter.

In [45]:
target = spark.createDataFrame([("T1",)], "trip_id string")                       # already processed
source = spark.createDataFrame(
    [("T1", "148.00"), ("T2", "abc"), ("T3", "95.50")], "trip_id string, fare string")

todo = source.join(target, "trip_id", "left_anti")                              # only T2, T3
clean = (todo
         .withColumn("fare", F.col("fare").try_cast("decimal(10,2)"))
         .filter(F.col("fare").isNotNull() & (F.col("fare") >= 0)))             # T2 fails
clean.show()

+-------+-----+
|trip_id| fare|
+-------+-----+
|     T3|95.50|
+-------+-----+



### Try it (exercise 5)

Make a "re-sent" DataFrame with the first 5 April rows plus one new trip id, and use a `left_anti` join to find the rows that are **not** already in `raw`.

In [46]:
# Exercise 5: your code here

## 20 · Reading PySpark errors

Spark errors are long because they cross Python and the JVM. Find two things:

1. **your last line** of Python code in the traceback;
2. the **error class** in square brackets, often with a suggestion.

| Error class | Typical cause | Fix |
|---|---|---|
| `UNRESOLVED_COLUMN` | column name typo | use a suggested name |
| `CANNOT_CONVERT_COLUMN_INTO_BOOL` | `and` / `or` / `not`, or missing parentheses | `&` `\|` `~` with parentheses |
| `CAST_INVALID_INPUT` | `cast()` on a malformed value (ANSI mode) | `try_cast`, `try_to_timestamp` |
| `PATH_ALREADY_EXISTS` | writing to an existing path | `mode("overwrite")` or `"append"` |

In [47]:
try:
    raw.select("fare")
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0][:230])

AnalysisException -> [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `fare` cannot be resolved. Did you mean one of the following? [`fare_thb`, `status`, `trip_id`, `driver_id`, `accepted_at`]. SQLSTATE: 42703;


---
## Wrap-up: where each concept appears in the PySpark tutorial

| Tutorial step (`tutorial_pyspark.ipynb`) | Concepts used |
|---|---|
| 0 · Setup: SparkSession, Arrow bridge | 2 SparkSession · 10 time zone · 16 Arrow · 5 shuffle partitions |
| 2 · Ingest CSV into bronze | 6 read CSV, binaryFile · 7 columns · 13 windows · 11 left_anti · 4 actions |
| 3 · Transform bronze → silver | 9 try_cast, try_to_timestamp · 10 to_date, hour · 7 when · 8 NULL filter · 13 row_number |
| 4, 7 · Show both layers | 14 temp views, spark.sql · 12 groupBy, count_if · 15 show |
| 5 · Append May without duplicates | 19 file guard, row guard · 11 left_anti |
| 8 · Snapshots | 18 lakehouse tables and snapshots |

Next: open **`tutorial_pyspark.ipynb`** in this folder.

---
# Solutions

Try the exercises yourself before reading these.

**Exercise 1:** price band for one zone, and NULL fares.

In [48]:
(raw.filter(F.col("service_zone") == "Ari")
    .withColumn("fare", F.col("fare_thb").cast("double"))
    .withColumn("price_band", F.when(F.col("fare") >= 170, "high")
                               .when(F.col("fare") >= 140, "mid")
                               .otherwise("low"))
    .groupBy("price_band").count().show())

raw.filter(F.col("fare_thb").isNull()).groupBy("status").count().show()

+----------+-----+
|price_band|count|
+----------+-----+
|      high|   23|
|       mid|    8|
|       low|    2|
+----------+-----+



+---------+-----+
|   status|count|
+---------+-----+
|cancelled|    8|
+---------+-----+



**Exercise 2:** average pickup wait per zone (DataFrame API).

In [49]:
wait = (raw
    .filter(F.col("status") == "completed")
    .withColumn("req", F.try_to_timestamp("requested_at", F.lit(TS)))
    .withColumn("pick", F.try_to_timestamp("picked_up_at", F.lit(TS)))
    .withColumn("wait_min", (F.unix_timestamp("pick") - F.unix_timestamp("req")) / 60))

(wait.groupBy("service_zone")
     .agg(F.round(F.avg("wait_min"), 2).alias("avg_pickup_wait_min"))
     .orderBy("service_zone").show())

+------------+-------------------+
|service_zone|avg_pickup_wait_min|
+------------+-------------------+
|         Ari|              10.26|
|        Asok|               9.83|
|        Siam|               9.78|
|       Silom|              10.39|
+------------+-------------------+



**Exercise 3:** latest request per driver.

In [50]:
latest_per_driver = Window.partitionBy("driver_id").orderBy(F.desc("requested_at"))
latest_trips = (raw
    .withColumn("rn", F.row_number().over(latest_per_driver))
    .filter("rn = 1").drop("rn"))
print(latest_trips.count(), "drivers")
latest_trips.select("driver_id", "trip_id", "requested_at").orderBy("driver_id").show(5, truncate=False)

18 drivers
+---------+---------------+-------------------------+
|driver_id|trip_id        |requested_at             |
+---------+---------------+-------------------------+
|D001     |BKK-20260429-E1|2026-04-29T18:03:00+07:00|
|D002     |BKK-20260429-E2|2026-04-29T18:20:00+07:00|
|D003     |BKK-20260430-M1|2026-04-30T08:03:00+07:00|
|D004     |BKK-20260430-M2|2026-04-30T08:20:00+07:00|
|D005     |BKK-20260430-M3|2026-04-30T08:37:00+07:00|
+---------+---------------+-------------------------+
only showing top 5 rows


**Exercise 4:** exercise 2 in Spark SQL.

In [51]:
spark.sql('''
  SELECT service_zone,
         round(avg(unix_timestamp(try_to_timestamp(picked_up_at, "yyyy-MM-dd'T'HH:mm:ssXXX"))
                 - unix_timestamp(try_to_timestamp(requested_at, "yyyy-MM-dd'T'HH:mm:ssXXX"))) / 60, 2)
           AS avg_pickup_wait_min
  FROM trips
  WHERE status = 'completed'
  GROUP BY service_zone
  ORDER BY service_zone''').show()

+------------+-------------------+
|service_zone|avg_pickup_wait_min|
+------------+-------------------+
|         Ari|              10.26|
|        Asok|               9.83|
|        Siam|               9.78|
|       Silom|              10.39|
+------------+-------------------+



**Exercise 5:** find the rows that are not loaded yet.

In [52]:
resent = raw.limit(5).unionByName(
    raw.limit(1).withColumn("trip_id", F.lit("BKK-20260431-NEW")))
resent.join(raw.select("trip_id"), "trip_id", "left_anti").select("trip_id", "status").show()

+----------------+---------+
|         trip_id|   status|
+----------------+---------+
|BKK-20260431-NEW|completed|
+----------------+---------+



In [53]:
spark.stop()